# 15.3 - Binary gingival detector error audit

Compare saved weights on locked D2 validation images. This diagnostic uses source severity labels and does not change model weights or thresholds.

## Verify inputs

Check the original split and label checksums and the collected checkpoints. Read only validation images. Use the installed local inference runtime and record its version. Identical positive boxes with conflicting source severity are counted once and reported as ambiguous.

In [1]:
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd
import torch, ultralytics
from ultralytics import YOLO
from dentalselfie.detection_audit import box_iou, match_boxes

root = Path.cwd()
while not (root / 'pyproject.toml').exists():
    if root == root.parent:
        raise RuntimeError('Project root not found')
    root = root.parent
report = root / 'docs/reports/gingivitis_binary'
split_path = root / 'data/manifests/gum.splits.csv'
lock = json.loads((root / 'data/manifests/gum.lock.json').read_text())
def sha(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()
assert sha(split_path) == lock['splits_sha256']
val = pd.read_csv(split_path).query("split == 'val'")
assert len(val) == 160 and val.sample_id.is_unique
baselines = pd.read_csv(report / 'baseline_results.csv')
images, annotations = {}, {}
conflicts = []
for row in val.itertuples():
    image, label = root / row.image, root / row.label
    assert sha(image) == row.sha256 and sha(label) == row.label_sha256
    images[row.sample_id] = image
    # Mirror duplicate removal after the original severity labels merge into one class.
    unique = {}
    for line in label.read_text().splitlines():
        grade, x, y, w, h = np.asarray(line.split(), dtype=np.float32)
        coordinates = tuple(map(float, (x, y, w, h)))
        key = (bool(grade), coordinates)
        unique.setdefault(key, set()).add(int(grade))
    conflicts.extend((row.sample_id, sorted(grades), coords) for (_, coords), grades in unique.items() if len(grades) > 1)
    annotations[row.sample_id] = [(next(iter(grades)) if len(grades) == 1 else 3, (x-w/2, y-h/2, x+w/2, y+h/2))
                                  for (_, (x,y,w,h)), grades in unique.items()]
print('Conflicting positive grades (reported as grade 3, ambiguous):', conflicts)
print('Runtime:', ultralytics.__version__, '| validation images:', len(images))
display(pd.Series([grade for rows in annotations.values() for grade, _ in rows]).value_counts().sort_index())
assert sum(grade > 0 for rows in annotations.values() for grade, _ in rows) == 1870

Conflicting positive grades (reported as grade 3, ambiguous): [('cd603e9e41d9ecdd34ca', [1, 2], (0.3603940010070801, 0.3620719909667969, 0.07622300088405609, 0.10199499875307083))]
Runtime: 8.4.183 | validation images: 160


0     49
1    931
2    938
3      1
Name: count, dtype: int64

## Run saved-weight inference

Use CPU with 640 input and NMS IoU 0.7. Match predictions at confidence 0.25 and box IoU 0.50. Keep a lower confidence pool for a fixed sensitivity check at 0.10 and 0.50. These operating points are diagnostic.

In [2]:
torch.set_num_threads(min(4, torch.get_num_threads()))
predictions = []
for model_row in baselines.itertuples():
    checkpoint = next((root / f'data/runs/gum_binary_{model_row.model.lower()[4:]}_v{model_row.version}').rglob('best.pt'))
    assert sha(checkpoint) == model_row.checkpoint_sha256
    model = YOLO(checkpoint)
    stream = model.predict(source=[str(path) for path in images.values()], stream=True, batch=8,
                           imgsz=640, conf=0.10, iou=0.70, max_det=300, device='cpu', verbose=False)
    seen = set()
    for result in stream:
        sample = Path(result.path).stem
        assert sample in images and sample not in seen
        seen.add(sample)
        for box, score in zip(result.boxes.xyxyn.cpu().numpy(), result.boxes.conf.cpu().numpy()):
            predictions.append((model_row.model, sample, float(score), *map(float, box)))
    assert seen == set(images)
    print(model_row.model, 'evaluated', len(seen), 'images', flush=True)
prediction_frame = pd.DataFrame(predictions, columns=['model','sample_id','confidence','x1','y1','x2','y2'])
prediction_frame.to_csv(report / 'audit_predictions.csv', index=False)

YOLOv8n evaluated 160 images


YOLOv8s evaluated 160 images


## Measure misses and grade-0 overlap

Assign each predicted box to at most one positive region. Count a grade-0 overlap only for a prediction left unmatched after positive matching. Overlap is defined at IoU 0.50. It measures agreement with the source annotations, with no whole-image negative cohort.

In [3]:
details, overlap_details, summary = [], [], []
coordinates = ['x1', 'y1', 'x2', 'y2']
for model_name in baselines.model:
    model_predictions = prediction_frame[prediction_frame.model == model_name]
    for threshold in [0.10, 0.25, 0.50]:
        for sample, rows in annotations.items():
            positive = [(grade, box) for grade, box in rows if grade > 0]
            negative = [box for grade, box in rows if grade == 0]
            subset = model_predictions[(model_predictions.sample_id == sample) & (model_predictions.confidence >= threshold)]
            boxes = subset[coordinates].to_numpy()
            events, missing = match_boxes([box for _, box in positive], boxes, subset.confidence.to_numpy())
            for index, (grade, box) in enumerate(positive):
                details.append((model_name, threshold, sample, grade, index not in missing, *box))
            unmatched = [event['prediction_index'] for event in events if event['gt_index'] is None]
            overlaps = box_iou(boxes[unmatched], negative) >= 0.50
            for region, box in enumerate(negative):
                overlap_details.append((model_name, threshold, sample, bool(overlaps[:, region].any()), *box))
            summary.append((model_name, threshold, sample, len(events), len(unmatched), int(overlaps.any(axis=1).sum())))
regions = pd.DataFrame(details, columns=['model','confidence','sample_id','grade','found',*coordinates])
negatives = pd.DataFrame(overlap_details, columns=['model','confidence','sample_id','overlapped',*coordinates])
per_image = pd.DataFrame(summary, columns=['model','confidence','sample_id','predicted','unmatched','grade0_overlap_predictions'])
recall = regions.groupby(['model','confidence','grade']).found.agg(regions='size', matched='sum', recall='mean')
display(recall)
if len(negatives):
    display(negatives.groupby(['model','confidence']).overlapped.agg(regions='size',overlapped='sum'))
display(per_image.groupby(['model','confidence'])[['predicted','unmatched','grade0_overlap_predictions']].sum())
for name, frame in [('audit_regions',regions),('audit_grade0_regions',negatives),('audit_images',per_image),('audit_recall',recall.reset_index())]:
    frame.to_csv(report / f'{name}.csv', index=False)

regions  matched    recall
model   confidence grade                            
YOLOv8n 0.10       1          931      911  0.978518
                   2          938      913  0.973348
                   3            1        1  1.000000
        0.25       1          931      908  0.975295
                   2          938      905  0.964819
                   3            1        1  1.000000
        0.50       1          931      869  0.933405
                   2          938      861  0.917910
                   3            1        1  1.000000
YOLOv8s 0.10       1          931      906  0.973147
                   2          938      915  0.975480
                   3            1        1  1.000000
        0.25       1          931      902  0.968851
                   2          938      904  0.963753
                   3            1        1  1.000000
        0.50       1          931      865  0.929108
                   2          938      864  0.921109
                   3            1        1  1.000000

regions  overlapped
model   confidence                     
YOLOv8n 0.10             49          49
        0.25             49          49
        0.50             49          45
YOLOv8s 0.10             49          49
        0.25             49          49
        0.50             49          43

predicted  unmatched  grade0_overlap_predictions
model   confidence                                                  
YOLOv8n 0.10             2358        533                          58
        0.25             2060        246                          50
        0.50             1836        105                          45
YOLOv8s 0.10             2293        471                          58
        0.25             1938        131                          49
        0.50             1826         96                          43